# Gold — Hospital Daily Summary

Builds `mano_adb_outlook.gold.hospital_daily_summary` — one row per hospital per calendar day with test volumes, abnormal/critical rates, and day-over-day trends.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# ----- Aggregate lab_results by hospital + test_date -----
base = (
    spark.table("mano_adb_outlook.silver.lab_results")
    .groupBy("hospital_id", "test_date")
    .agg(
        count("*").alias("total_tests"),
        countDistinct("patient_id").alias("unique_patients_tested"),
        countDistinct("doctor_id").alias("active_doctor_count"),
        count(when(col("result_status") == "Abnormal", True)).alias("abnormal_test_count"),
        count(when(col("result_status") == "Critical", True)).alias("critical_test_count"),
    )
    .withColumnRenamed("test_date", "summary_date")
)

# ----- Join hospitals & calendar -----
hospitals = spark.table("mano_adb_outlook.silver.hospitals").select("hospital_id", "hospital_name")
calendar = spark.table("mano_adb_outlook.silver.Calendar").select("date_key", "day_name", "is_weekend")

df = (
    base.alias("b")
    .join(hospitals.alias("h"), "hospital_id", "left")
    .join(
        calendar.alias("c"),
        date_format(col("b.summary_date"), "yyyyMMdd").cast("int") == col("c.date_key"),
        "left",
    )
)

# ----- Derived rates -----
df = (
    df
    .withColumn("abnormal_rate_pct", round(col("abnormal_test_count") / col("total_tests") * 100, 2))
    .withColumn("critical_rate_pct", round(col("critical_test_count") / col("total_tests") * 100, 2))
)

# ----- Window: day-over-day trend -----
w = Window.partitionBy("hospital_id").orderBy("summary_date")
df = (
    df
    .withColumn("prev_day_total_tests", lag("total_tests").over(w))
    .withColumn(
        "test_volume_change_pct",
        when(
            (col("prev_day_total_tests").isNull()) | (col("prev_day_total_tests") == 0), None
        ).otherwise(
            round((col("total_tests") - col("prev_day_total_tests")) / col("prev_day_total_tests") * 100, 2)
        ),
    )
)

# ----- Select final columns -----
df_final = df.select(
    "hospital_id", "hospital_name", "summary_date",
    col("c.day_name").alias("day_name"),
    col("c.is_weekend").alias("is_weekend"),
    "total_tests", "unique_patients_tested", "active_doctor_count",
    "abnormal_test_count", "critical_test_count",
    col("abnormal_rate_pct").cast("decimal(5,2)"),
    col("critical_rate_pct").cast("decimal(5,2)"),
    "prev_day_total_tests",
    col("test_volume_change_pct").cast("decimal(6,2)"),
)

# ----- Write to gold -----
df_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("mano_adb_outlook.gold.hospital_daily_summary")

In [0]:
row_count = spark.table("mano_adb_outlook.gold.hospital_daily_summary").count()
dbutils.notebook.exit(str(row_count))